# Notebook Demo: Structure & Usage Mining: Link Analysis, Random Walks, and PageRank

### Demo Highlights

This notebook demonstrates key ideas in **web structure mining and usage mining** using Python.

Students will learn how to:

- Represent the web as a **directed graph**
- Analyze **link structure** (in-degree, out-degree)
- Understand the **random surfer model**
- Implement **PageRank from scratch**
- Compute PageRank using **NetworkX**
- Interpret **page importance from link structure**


**Goal:** understand how hyperlink structure and navigation behavior can be used to rank and analyze web pages.


---

In [ ]:
!pip install networkx

In [ ]:
# Imports
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
pd.set_option("display.precision", 4)

## 1: Create a Toy Web Graph

We begin with a small directed graph representing a miniature web.

Interpretation:
- each node is a web page
- each arrow is a hyperlink
- some pages may receive many links
- some pages may act as hubs by linking to many others

This kind of toy example is useful because we can inspect all relationships directly before moving to larger graphs.

In [ ]:
# Define a toy directed web graph
pages = ["Home", "About", "Products", "Blog", "Support", "Contact"]

edges = [
    ("Home", "About"),
    ("Home", "Products"),
    ("Home", "Blog"),
    ("About", "Home"),
    ("Products", "Home"),
    ("Products", "Support"),
    ("Products", "Contact"),
    ("Blog", "Home"),
    ("Blog", "Products"),
    ("Blog", "Support"),
    ("Support", "Contact"),
    ("Contact", "Home"),
]

G = nx.DiGraph()
G.add_nodes_from(pages)
G.add_edges_from(edges)

print("Nodes:", list(G.nodes()))
print("Edges:", list(G.edges()))

## 2: Visualize the Directed Graph

A graph picture helps students connect the abstract matrix form to the web structure.

When reading the plot, ask:

- Which pages appear to receive many incoming links?
- Which pages send links to many others?
- Which pages may become important by being connected to strong pages?

In [ ]:
plt.figure(figsize=(8, 6))
pos = nx.spring_layout(G, seed=7)
nx.draw_networkx(
    G, pos,
    with_labels=True,
    node_size=2200,
    arrows=True,
    font_size=10
)
plt.title("Toy Web Graph")
plt.axis("off")
plt.show()

## 3: Basic Structure-Mining Statistics

Before PageRank, we can inspect simple graph statistics:

- **in-degree**: number of incoming links
- **out-degree**: number of outgoing links

These are not the same as PageRank, but they provide a baseline.

A page with high in-degree may be popular.  
A page with high out-degree may act like a hub.

In [ ]:
structure_df = pd.DataFrame({
    "page": pages,
    "in_degree": [G.in_degree(p) for p in pages],
    "out_degree": [G.out_degree(p) for p in pages],
})

structure_df.sort_values("in_degree", ascending=False)

## 4: Build the Adjacency Matrix

The **adjacency matrix** is a numerical representation of the directed graph.

If page `i` links to page `j`, then:

- `A[i, j] = 1`
- otherwise `A[i, j] = 0`

This matrix is the starting point for many graph algorithms.

In [ ]:
A = nx.to_numpy_array(G, nodelist=pages, dtype=int)
adj_df = pd.DataFrame(A, index=pages, columns=pages)
adj_df

## 5: Convert the Graph into a Transition Matrix

For random-walk ranking, we need a **transition matrix**.

For each page:
- divide each outgoing link by the number of outgoing links
- this creates probabilities of moving to linked pages

So if a page has 3 outgoing links, each linked page gets probability `1/3`.

In [ ]:
n = len(pages)
P = np.zeros((n, n), dtype=float)

for i, page in enumerate(pages):
    out_links = list(G.successors(page))
    if len(out_links) > 0:
        prob = 1 / len(out_links)
        for target in out_links:
            j = pages.index(target)
            P[i, j] = prob

transition_df = pd.DataFrame(P, index=pages, columns=pages)
transition_df

## 6: Check Row Sums of the Transition Matrix

Each row of a valid transition matrix should sum to 1.

That means:
- from every current page,
- the surfer must move somewhere with total probability 1

This is a basic sanity check for Markov-style models.

In [ ]:
row_sums = transition_df.sum(axis=1)
row_sums

## 7: First Intuition — Ranking by In-Degree Only

Before PageRank, let us rank pages by a very simple metric: incoming link count.

This helps show why PageRank is more sophisticated.

A page may receive many links, but:
- are those linking pages important?
- do they split their vote among many pages?
- are users actually navigating there?

In [ ]:
indegree_rank = structure_df[["page", "in_degree"]].sort_values("in_degree", ascending=False)
indegree_rank.reset_index(drop=True)

## 8: Implement PageRank from Scratch

We now implement the iterative PageRank algorithm.

Core idea:
- start with equal probability on all pages
- repeatedly update scores
- each page distributes its score across outgoing links
- teleportation prevents dead ends and improves convergence

Update formula in words:

new score of a page =
- random jump contribution
- plus weighted contributions from linking pages

In [ ]:
def pagerank_manual(graph, nodes, d=0.85, max_iter=100, tol=1e-8):
    n = len(nodes)
    rank = np.ones(n) / n

    # build row-stochastic transition matrix with sink handling
    P = np.zeros((n, n), dtype=float)
    index = {node: i for i, node in enumerate(nodes)}

    for i, node in enumerate(nodes):
        out_neighbors = list(graph.successors(node))
        if len(out_neighbors) == 0:
            # sink node: distribute uniformly
            P[i, :] = 1 / n
        else:
            prob = 1 / len(out_neighbors)
            for nbr in out_neighbors:
                j = index[nbr]
                P[i, j] = prob

    history = [rank.copy()]

    for _ in range(max_iter):
        new_rank = (1 - d) / n + d * (rank @ P)
        history.append(new_rank.copy())

        if np.linalg.norm(new_rank - rank, ord=1) < tol:
            rank = new_rank
            break
        rank = new_rank

    return rank, np.array(history), P

rank_manual, history, P_sink_handled = pagerank_manual(G, pages, d=0.85)
pd.DataFrame({"page": pages, "pagerank_manual": rank_manual}).sort_values("pagerank_manual", ascending=False)

## 9: Monitor Convergence Across Iterations

PageRank is not obtained in one step.  
It is found by repeated updates until the scores stabilize.

This plot helps students see:
- some pages gain importance gradually
- others lose importance
- the process converges to a stable ranking

In [ ]:
plt.figure(figsize=(9, 5))
for i, page in enumerate(pages):
    plt.plot(history[:, i], label=page)

plt.xlabel("Iteration")
plt.ylabel("PageRank score")
plt.title("Convergence of Manual PageRank")
plt.legend()
plt.grid(True)
plt.show()

## 10: Compare with NetworkX PageRank

To verify our implementation, we compare our manual scores with a trusted library implementation.

Small numerical differences may appear because of:
- stopping criteria
- numerical precision
- implementation details

But the ranking order should be very similar.

In [ ]:
pr_nx = nx.pagerank(G, alpha=0.85)

compare_df = pd.DataFrame({
    "page": pages,
    "pagerank_manual": rank_manual,
    "pagerank_networkx": [pr_nx[p] for p in pages]
})

compare_df["abs_diff"] = np.abs(compare_df["pagerank_manual"] - compare_df["pagerank_networkx"])
compare_df.sort_values("pagerank_networkx", ascending=False)

## 11: Add a Sink Node to Illustrate a Practical Issue

A **sink node** is a page with no outgoing links.

Examples in practice:
- a PDF page
- a terminal article page
- a thank-you page
- a dead-end information page

Without special treatment, probability mass can get trapped there.

In [ ]:
pages2 = pages + ["Archive"]

G2 = G.copy()
G2.add_node("Archive")
G2.add_edge("Blog", "Archive")
G2.add_edge("Support", "Archive")
# Archive intentionally has no outgoing links

print("Out-degree of Archive:", G2.out_degree("Archive"))

## 12: Recompute PageRank with Sink Handling

Our manual implementation already handles sink nodes by redistributing sink probability uniformly.

This is effectively equivalent to saying:

> if the surfer reaches a dead end, they may jump to any page uniformly.

In [ ]:
rank_manual_2, history2, P2 = pagerank_manual(G2, pages2, d=0.85)
result2 = pd.DataFrame({
    "page": pages2,
    "pagerank_with_sink": rank_manual_2
}).sort_values("pagerank_with_sink", ascending=False)
result2.reset_index(drop=True)

## 13: Explore the Effect of the Damping Factor

The damping factor `d` controls how strongly the surfer follows links.

Typical interpretation:
- larger `d`: structure matters more
- smaller `d`: random jumping matters more

We compare multiple damping values to show sensitivity.

In [ ]:
d_values = [0.50, 0.70, 0.85, 0.95]
rank_by_d = {"page": pages2}

for d in d_values:
    r, _, _ = pagerank_manual(G2, pages2, d=d)
    rank_by_d[f"d={d}"] = r

damping_df = pd.DataFrame(rank_by_d)
damping_df.sort_values("d=0.85", ascending=False)

## 14: Structure Mining Interpretation

At this point, discuss questions such as:

- Why might `Home` or `Products` receive strong scores?
- Why is PageRank different from simple in-degree?
- How does being linked from an important page help?

This is the key teaching moment:
PageRank captures **quality of incoming links**, not just quantity.

In [ ]:
# Visual comparison: in-degree vs PageRank
merged = structure_df.merge(
    pd.DataFrame({"page": pages, "pagerank": rank_manual}),
    on="page"
)

merged = merged.sort_values("pagerank", ascending=False)

plt.figure(figsize=(8,5))
x = np.arange(len(merged))
width = 0.38

plt.bar(x - width/2, merged["in_degree"], width=width, label="In-degree")
plt.bar(x + width/2, merged["pagerank"], width=width, label="PageRank")

plt.xticks(x, merged["page"], rotation=30)
plt.ylabel("Score")
plt.title("Simple Link Count vs PageRank")
plt.legend()
plt.tight_layout()
plt.show()

merged

## 15: Shift to Usage Mining — Simulated Navigation Logs

Now we move from link structure to **user behavior**.

Suppose we observe clickstream sessions such as:

- Home → Products → Support → Contact
- Home → Blog → Products
- Blog → Home → About

These transitions reveal how users actually move through the site.

In practice, such data may come from:
- web server logs
- analytics platforms
- event tracking systems

In [ ]:
# Simulated clickstream sessions
sessions = [
    ["Home", "Products", "Support", "Contact"],
    ["Home", "Blog", "Products", "Contact"],
    ["Blog", "Home", "About"],
    ["Home", "Products", "Home", "Blog"],
    ["Products", "Support", "Archive"],
    ["Home", "Blog", "Archive"],
    ["Contact", "Home", "Products"],
    ["Home", "About", "Home", "Products"],
    ["Blog", "Products", "Support"],
    ["Home", "Products", "Contact", "Home"],
]

sessions

## 16: Build a Usage Transition Matrix from Click Data

From session logs, we count observed page-to-page transitions.

This creates a **usage-based transition matrix**:

- rows = current page
- columns = next page
- values = empirical probabilities estimated from observed clicks

This differs from structure-based transitions because it reflects actual behavior, not just available links.

In [ ]:
usage_nodes = pages2
counts = pd.DataFrame(0, index=usage_nodes, columns=usage_nodes, dtype=int)

for session in sessions:
    for a, b in zip(session[:-1], session[1:]):
        if a in counts.index and b in counts.columns:
            counts.loc[a, b] += 1

counts

In [ ]:
usage_transition = counts.div(counts.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)
usage_transition

## 17: Estimate a Usage-Based Stationary Distribution

If users continue navigating according to observed transition probabilities, what long-run page visit distribution emerges?

This gives a simple **usage-based ranking**.  
It is not exactly PageRank, but it is closely related to Markov-chain behavior.

In [ ]:
def stationary_distribution(P, max_iter=200, tol=1e-10):
    n = P.shape[0]
    P = P.copy().astype(float)
    zero_rows = np.where(P.sum(axis=1) == 0)[0]
    for i in zero_rows:
        P[i, :] = 1 / n

    x = np.ones(n) / n
    history = [x.copy()]

    for _ in range(max_iter):
        x_new = x @ P
        history.append(x_new.copy())
        if np.linalg.norm(x_new - x, ord=1) < tol:
            x = x_new
            break
        x = x_new

    return x, np.array(history), P

usage_rank, usage_hist, usage_P = stationary_distribution(usage_transition.values)

usage_rank_df = pd.DataFrame({
    "page": usage_nodes,
    "usage_rank": usage_rank
}).sort_values("usage_rank", ascending=False)

usage_rank_df.reset_index(drop=True)

## 18: Compare Structure-Based Ranking and Usage-Based Ranking

Now we compare:

- **PageRank** from the hyperlink structure
- **usage rank** from observed click behavior

This comparison is pedagogically useful because it shows that:

- a page may be important structurally
- but not heavily used in practice

or the reverse.

In [ ]:
pr2_nx = nx.pagerank(G2, alpha=0.85)

comparison = pd.DataFrame({
    "page": usage_nodes,
    "pagerank": [pr2_nx[p] for p in usage_nodes],
    "usage_rank": [usage_rank[usage_nodes.index(p)] for p in usage_nodes],
    "in_degree": [G2.in_degree(p) for p in usage_nodes],
    "visit_count": [sum(page == p for sess in sessions for page in sess) for p in usage_nodes]
})

comparison.sort_values("pagerank", ascending=False)

In [ ]:
comparison_plot = comparison.set_index("page")[["pagerank", "usage_rank"]].sort_values("pagerank", ascending=False)

comparison_plot.plot(kind="bar", figsize=(9,5))
plt.title("Structure-Based vs Usage-Based Ranking")
plt.ylabel("Score")
plt.xticks(rotation=30)
plt.grid(axis="y")
plt.tight_layout()
plt.show()

## 19: Discussion — Why Can the Rankings Differ?

Structure mining and usage mining answer different questions.

### Structure mining asks:
Which pages are important according to the link graph?

### Usage mining asks:
Which pages are important according to observed user behavior?

Differences may arise because:
- users do not click all available links equally
- some pages are hidden behind menus or search
- certain pages attract repeated visits
- structural importance does not always mean practical popularity

In [ ]:
# Correlation-style view
comparison[["pagerank", "usage_rank", "in_degree", "visit_count"]].corr()

## External Theory Note — Ranking as a Markov Chain Problem

Many ranking methods can be interpreted through **Markov chains**.

A transition matrix describes how probability moves from one state to another.

In web ranking:
- a state is a page
- a transition is a click or hyperlink traversal
- a ranking score can be interpreted as long-run visit probability

Why this matters in data mining:
- it connects web mining to probability, graphs, and linear algebra
- it provides interpretable and scalable ranking methods
- it supports search, recommendation, and anomaly detection

The key intuition is:

> importance can be modeled as the amount of probability that accumulates on a node over repeated transitions.

## Final Takeaways

In this demo, we studied two complementary forms of web mining.

### Structure mining
Uses the link graph itself.
- pages are nodes
- links are directed edges
- PageRank propagates importance through the graph

### Usage mining
Uses user interaction data.
- sessions reveal behavioral preferences
- transitions estimate empirical navigation probabilities
- rankings can differ from structure-only methods

### Key lesson
A strong web analytics system often benefits from **both**:
- structural signals for authority and connectivity
- behavioral signals for actual user engagement

In advanced data mining, these ideas extend to:
- citation networks
- recommendation graphs
- social influence analysis
- knowledge graphs
- graph neural networks

# See you next lecture!